# 00 · Colab kurulumu ve deney koşuları

Bu notebook depoyu Colab'a kurar, testleri çalıştırır ve deneyleri başlatır.

1. **Çalışma zamanı:** Menüden *Runtime → Change runtime type → T4 GPU* seç. Aylık deney CPU'da da hızlıdır; günlük deneyler GPU ile çok daha kısa sürer.
2. Hücreleri yukarıdan aşağıya sırayla çalıştır.
3. Colab oturumu kapanınca `/content` altındaki her şey silinir. Sonuçları saklamak için en alttaki **Drive'a kopyala** hücresini çalıştır.

## 1. Depoyu getir

In [ ]:
import os

REPO_URL = "https://github.com/salihcengiz/sector-spillover.git"
ROOT = "/content/sector-spillover"

if os.path.isdir(ROOT):
    !git -C {ROOT} pull --ff-only     # depo zaten varsa son sürümü çek
else:
    !git clone {REPO_URL} {ROOT}
%cd {ROOT}
!git log --oneline -3

## 2. Kurulum

torch, numpy, pandas, scikit-learn, scipy, matplotlib, pyyaml ve pytest Colab'da hazır geliyor. Yalnızca projenin kendi paketi (`spillover`) kuruluyor; `requirements.txt` bilerek kurulmuyor.

In [ ]:
!pip install -q -e .

import sys, numpy, pandas, torch, yaml
print("python ", sys.version.split()[0])
print("numpy  ", numpy.__version__, "| pandas", pandas.__version__, "| torch", torch.__version__)
print("cihaz  ", "cuda: " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu (GPU seçilmemiş)")

## 3. Testler

Testlerin hepsi geçmeli. Model ve eğitim testleri (`test_models.py`, `test_experiment.py` içindeki LSTM testi) ilk kez burada torch ile çalışıyor.

In [ ]:
!python -m pytest -q

## 4. Hızlı deneme (`--quick`: tek seed, en fazla 2 epoch)

Uçtan uca akışın hatasız çalıştığını birkaç dakikada doğrular. Bu koşunun sonuçları anlamlı değildir.

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_monthly_b.yaml --quick

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_daily_crisis.yaml --quick --folds covid

## 5. Tam koşular

Her model 5 seed ile eğitilir ve seed'lerin ortalaması da ayrı bir model olarak raporlanır.

- `lstm_monthly_b`: aylık, B düzeni (train ≤ 2010, val 2011–2016, test 2017 → 2026-09).
- `lstm_daily_crisis`: günlük, hocanın düzeni. COVID, SVB ve son 1 yıl için üç ayrı katman; her katmanda modeller yeniden eğitilir.
- `lstm_daily_b`: günlük, B düzeni. Ön analizle karşılaştırmak için, isteğe bağlı.

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_monthly_b.yaml

In [ ]:
!python scripts/run_experiment.py --config configs/lstm_daily_crisis.yaml

In [ ]:
# İsteğe bağlı:
# !python scripts/run_experiment.py --config configs/lstm_daily_b.yaml

## 6. Sonuçlara hızlı bakış

Her koşunun klasörü `outputs/runs/<zaman>_<ad>/`. Aşağıdaki hücre, adı verilen deneyin en son tam koşusunu gösterir.

- `oos_r2`: train ortalamasına göre test R² (49 sektörün ortalaması).
- `rel_mse`: HAR-kendi'ye göre göreli MSE; 1'in altı daha iyi demek.
- DM tablosunda istatistik negatifse ilk model daha iyidir.

In [ ]:
import glob
import pandas as pd

def latest_run(name):
    runs = sorted(r for r in glob.glob(f"outputs/runs/*_{name}") if not r.endswith("_quick"))
    if not runs:
        raise FileNotFoundError(f"{name} için tam koşu bulunamadı.")
    return runs[-1]

def show(name):
    run = latest_run(name)
    print(run)
    summary = pd.read_csv(f"{run}/summary_all.csv")
    display(summary.pivot(index="model", columns="fold", values="oos_r2").round(3).rename_axis(columns="oos_r2"))
    display(summary.pivot(index="model", columns="fold", values="rel_mse").round(3).rename_axis(columns="rel_mse"))
    rows = []
    for fold in summary["fold"].unique():
        dm = pd.read_csv(f"{run}/{fold}/dm.csv")
        rows.append(dm[(dm.sector == "ALL") & (dm.loss == "mse")].assign(fold=fold))
    display(pd.concat(rows)[["fold", "model", "reference", "stat", "p_value"]].round(4))

show("lstm_monthly_b")

In [ ]:
show("lstm_daily_crisis")

## 7. Sonuçları Google Drive'a kopyala

Drive'da `MyDrive/sector-spillover/outputs/` klasörüne kopyalar. Daha önce kopyalanmış koşuların üzerine yazmaz.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/sector-spillover/outputs
!rsync -a --ignore-existing outputs/ /content/drive/MyDrive/sector-spillover/outputs/
!ls /content/drive/MyDrive/sector-spillover/outputs/runs